# Kokoro TTS Fine-Tuning Notebook (Google Colab / Kaggle T4)

This notebook fine-tunes **Kokoro-82M** (StyleTTS2 architecture) on custom emotional speech datasets and exports an optimized, quantized ONNX model for CPU inference.

### Pipeline Overview:
1. **Environment & GPU Setup** (Free T4 or better)
2. **Clone Training Harness (`kikiri-tts`) & Install Dependencies**
3. **Download Base Kokoro Checkpoint**
4. **Stage 1: Acoustic Training** (Duration, F0, Mel loss)
5. **Stage 2: Adversarial Training** (WavLM perceptual loss, MPD, MRD)
6. **Extract Emotion Voicepacks (`.npy`)**
7. **Export & Quantize to INT8 ONNX**

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive (Ensures checkpoints are saved if Colab disconnects)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kokoro_training/checkpoints

In [ ]:
# Step 3: Clone the verified Kokoro training harness and install dependencies
!git clone https://github.com/semidark/kikiri-tts.git /content/kikiri-tts
%cd /content/kikiri-tts
!pip install -q torchaudio transformers soundfile misaki phonemizer onnx onnxruntime
!apt-get install -y espeak-ng

In [ ]:
# Step 4: Download base Kokoro-82M model checkpoint
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(repo_id="hexgrad/Kokoro-82M", filename="kokoro-v1_0.pth", local_dir="/content/kikiri-tts/models")
print("Downloaded base model to:", model_path)

### Step 5: Upload Dataset Manifests
Upload your `train_list.txt` and `val_list.txt` created with `python training/phonemize_dataset.py`.

Upload `conditioning.jsonl` alongside the manifests. Before running the upstream `train_first.py` / `train_second.py` harness, patch its dataset loader to read this sidecar by audio path, feed the duration targets and emotion ID to the duration predictor, and add `0.1 * style_classification_loss(...)` from `training.emotion_conditioning` to each generator objective. Keep the text encoder, duration predictor, emotion embedding, and style projection in FP16/FP32.


In [ ]:
# Step 6: Stage 1 Training (Acoustic Pretraining)
# Freezes iSTFTNet vocoder; trains TextEncoder and ProsodyPredictor
!python train_first.py \
  --config_path Configs/config.yml \
  --train_data train_list.txt \
  --val_data val_list.txt \
  --pretrained_model models/kokoro-v1_0.pth \
  --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage1 \
  --batch_size 4

In [ ]:
# Step 7: Stage 2 Training (Adversarial Training with WavLM Perceptual Loss)
!python train_second.py \
  --config_path Configs/config.yml \
  --train_data train_list.txt \
  --val_data val_list.txt \
  --pretrained_model /content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth \
  --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage2 \
  --batch_size 4

In [ ]:
# Step 8: Export Fine-Tuned PyTorch Model with float conditioning heads
!python training/export_onnx.py \
  --model /content/drive/MyDrive/kokoro_training/checkpoints/stage2/best_model.pth \
  --out_dir /content/drive/MyDrive/kokoro_training/models \
  --conditioning_nodes style_projection,duration_predictor,emotion_embedding

# Verify the final graph: style/duration/emotion conditioning nodes must remain FP16/FP32.
